## Predictive Machine Learning Modeling

---
## Step 1 - Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score, confusion_matrix
from sklearn.calibration import calibration_curve
import statsmodels.api as sm
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
%matplotlib inline

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

TITLE_TEAMS = ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']

TEAM_PALETTE = {
    'Arsenal':            '#EF0107',
    'Liverpool':          '#00B2A9',
    'Manchester City':    '#6CABDD',
    'Manchester United':  '#FFB81C',
}

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

print('Imports ready')

Imports ready


---
## Step 2 - Load Data

In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PROC_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(PROC_DATA_DIR / 'all_4teams_processed.csv')
df['date'] = pd.to_datetime(df['date'])
df['season'] = df['season'].astype(str)
df = df.sort_values(['team', 'date']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Teams: {sorted(df["team"].unique())}')

Shape: (1064, 37)
Teams: ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United']


---
## Section A - Target Formulation

In [3]:
# is_home from venue, y from result (L/D/W -> 0/1/2)
# class balance check, this is the accuracy floor to beat later
df['is_home'] = (df['venue'] == 'home').astype(int)

result_order = ['L', 'D', 'W']
y_map = {'L': 0, 'D': 1, 'W': 2}
df['y'] = df['result'].map(y_map)

print('Class balance, all 1,064 matches:')
print(df['result'].value_counts().reindex(result_order))
print()
print('As a percentage:')
print((df['result'].value_counts(normalize=True).reindex(result_order) * 100).round(1))

Class balance, all 1,064 matches:
result
L    214
D    222
W    628
Name: count, dtype: int64

As a percentage:
result
L    20.1
D    20.9
W    59.0
Name: proportion, dtype: float64


### Reading Section A

---
## Section B - Feature Matrix, Part 1: Existing Features

In [4]:
# recap: xG_roll5, xGA_roll5, pts_roll5, win_rate_roll5, stakes_intensity,
# is_home, is_rivalry (adjusted in Section D)
# do NOT include is_high_stakes / is_high_stakes_retro as separate features,
# stakes_intensity is the continuous version and retro leaks future info


---
## Section C - Feature Matrix, Part 2: Opponent Quality

In [5]:
# reload full 20-team Understat schedule from local cache (sd.Understat),
# same trick NB02 used for title_gap, no new scraping
import soccerdata as sd

PL = 'ENG-Premier League'
SEASONS_STR = ['1920', '2021', '2122', '2223', '2324', '2425', '2526']

understat = sd.Understat(leagues = PL, seasons = SEASONS_STR)
full_schedule = understat.read_schedule().reset_index()
print(f'Full 20-team schedule loaded from cache: {full_schedule.shape}')

[09/17/26 16:36:47] INFO     No custom team name replacements found. You can configure these in       ]8;id=1182586;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=1182587;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py#91\91]8;;\
                             C:\Users\tejas\soccerdata\config\teamname_replacements.json.                          

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=1182593;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py\_config.py]8;;\:]8;id=1182594;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_config.py#189\189]8;;\
                             C:\Users\tejas\soccerdata\config\league_dict.json.                                    

                    INFO     Saving cached data to C:\Users\tejas\soccerdata\data\Understat          ]8;id=1182601;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=1182602;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\soccerdata\_common.py#250\250]8;;\

[2026-09-17 16:36:47] INFO     TLSLibrary:_load_library:397 - Successfully loaded TLS library: C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\bin\tls-client-xgo-1.13.1-windows-amd64.dll


                    INFO     Successfully loaded TLS library:                                      ]8;id=1182609;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py\libraries.py]8;;\:]8;id=1182610;file://C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\models\libraries.py#397\397]8;;\
                             C:\Users\tejas\anaconda3\envs\arsenal\Lib\site-packages\tls_requests\                 
                             bin\tls-client-xgo-1.13.1-windows-amd64.dll                                           

Full 20-team schedule loaded from cache: (2660, 20)


In [6]:
# wide-to-long reshape, this time keeping home_xg/away_xg as xG/xGA
KEEP = ['league', 'season', 'game_id', 'date', 'team', 'scored', 'conceded']
full_home = full_schedule.rename(columns = {
    'home_team': 'team', 'home_goals': 'scored', 'away_goals':'conceded',
    'home_xg': 'xG', 'away_xg': 'xGA'
})
full_away = full_schedule.rename(columns={
    'away_team': 'team', 'away_goals': 'scored', 'home_goals': 'conceded',
    'away_xg': 'xG', 'home_xg': 'xGA',
})

KEEP2 = KEEP + ['xG', 'xGA']

full_long = pd.concat([full_home[KEEP2], full_away[KEEP2]], ignore_index=True)
full_long['date'] = pd.to_datetime(full_long['date'])
full_long = full_long.sort_values(['team','season','date']).reset_index(drop = True)

print(f'Full 20-team long format: {full_long.shape}  (20 teams x 7 seasons x 38 = {20*7*38})')

Full 20-team long format: (5320, 9)  (20 teams x 7 seasons x 38 = 5320)


In [7]:
# opponent's own rolling xG/xGA, shift(1) before rolling, same anti-leakage
# rule as NB02's own rolling features
g = full_long.groupby(['team','season'])
full_long['opp_xG_roll5'] = g['xG'].transform(lambda x: x.shift(1).rolling(5, min_periods=3).mean())
full_long['opp_xGA_roll5'] = g['xGA'].transform(lambda x: x.shift(1).rolling(5, min_periods = 3).mean())

opp_feats = full_long[['game_id', 'team', 'opp_xG_roll5', 'opp_xGA_roll5']].rename(
    columns = {'team': 'opponent'}
)

In [8]:
# merge onto main df by game_id + opponent, build opp_xgd_roll5
# check null count, should only be the same early-season gap as our own
# rolling features
df = df.merge(opp_feats, on=['game_id', 'opponent'], how='left')
df['opp_xgd_roll5'] = df['opp_xG_roll5'] - df['opp_xGA_roll5']

print(f'Nulls in opp_xgd_roll5: {df["opp_xgd_roll5"].isna().sum()} / {len(df)}')
print('(nulls expected only for each opponent\'s own first 2 games of a season, same reason our own rolling features have early-season NaN)')

Nulls in opp_xgd_roll5: 83 / 1064
(nulls expected only for each opponent's own first 2 games of a season, same reason our own rolling features have early-season NaN)


In [9]:
# spot check one match by hand before trusting the merge at scale
spot_check = df[(df['team'] == 'Arsenal') & (df['season'] == '1920') & (df['gameweek'] == 3)]
print(spot_check[['date', 'opponent', 'venue', 'result', 'opp_xG_roll5', 'opp_xGA_roll5', 'opp_xgd_roll5']].to_string(index=False))

               date  opponent venue result  opp_xG_roll5  opp_xGA_roll5  opp_xgd_roll5
2019-08-24 17:30:00 Liverpool  away      L           NaN            NaN            NaN


### Reading Section C

---
## Section D - Big 6 vs Rivalry (fixing a double-count)

In [10]:
# is_big6_opp from a fixed Big 6 list
# check overlap between is_rivalry and is_big6_opp before deciding anything
BIG6 = ['Arsenal', 'Liverpool', 'Manchester City', 'Manchester United', 'Chelsea', 'Tottenham']
df['is_big6_opp'] = df['opponent'].isin(BIG6).astype(int)

overlap = pd.crosstab(df['is_rivalry'], df['is_big6_opp'])
overlap.index = ['not rivalry', 'rivalry']
overlap.columns = ['not big6', 'big6']
print(overlap)

             not big6  big6
not rivalry       770   112
rivalry            14   168


In [11]:
# is_non_big6_rivalry = is_rivalry AND NOT is_big6_opp
# confirm zero overlap with is_big6_opp
df['is_non_big6_rivalry'] = ((df['is_rivalry'] == True) & (df['is_big6_opp'] == 0)).astype(int)

print(f'is_big6_opp        : {df["is_big6_opp"].sum()} matches')
print(f'is_non_big6_rivalry: {df["is_non_big6_rivalry"].sum()} matches')
print(f'Overlap check (must be 0): {((df["is_big6_opp"]==1) & (df["is_non_big6_rivalry"]==1)).sum()}')
print()
print(df[df['is_non_big6_rivalry']==1].groupby(['team','opponent']).size())

is_big6_opp        : 280 matches
is_non_big6_rivalry: 14 matches
Overlap check (must be 0): 0

team       opponent
Liverpool  Everton     14
dtype: int64


### Reading Section D

---
## Section E - Assembling the Feature Matrix

In [12]:
# FEATURES list, drop rows with NaN rolling features
# recency_weight goes in as sample_weight at fit time, NOT as an X column
FEATURES = [
    'xG_roll5', 'xGA_roll5', 'pts_roll5', 'win_rate_roll5',
    'stakes_intensity', 'is_home',
    'is_big6_opp', 'is_non_big6_rivalry', 'opp_xgd_roll5',
]

model_df = df.dropna(subset=FEATURES + ['y']).copy()
print(f'Rows after dropping early-season NaN rolling features: {len(model_df)} / {len(df)}')
print()
print('Feature summary:')
print(model_df[FEATURES].describe().round(3).T)

Rows after dropping early-season NaN rolling features: 979 / 1064

Feature summary:
                     count   mean    std    min    25%    50%    75%    max
xG_roll5             979.0  1.968  0.498  0.590  1.591  1.949  2.297  3.714
xGA_roll5            979.0  1.174  0.448  0.285  0.862  1.107  1.431  3.334
pts_roll5            979.0  1.969  0.626  0.000  1.600  2.000  2.400  3.000
win_rate_roll5       979.0  0.586  0.236  0.000  0.400  0.600  0.800  1.000
stakes_intensity     979.0  0.396  0.282  0.000  0.161  0.349  0.588  1.000
is_home              979.0  0.502  0.500  0.000  0.000  1.000  1.000  1.000
is_big6_opp          979.0  0.260  0.439  0.000  0.000  0.000  1.000  1.000
is_non_big6_rivalry  979.0  0.014  0.119  0.000  0.000  0.000  0.000  1.000
opp_xgd_roll5        979.0 -0.005  0.810 -2.272 -0.568 -0.064  0.542  2.392


---
## Section F - Time-Series Train/Test Split

In [13]:
# headline split: train seasons <= 2024-25, test = 2025-26
SEASON_ORDER = [1920,2021,2122,2223,2324,2425,2526]
model_df['season_int'] = model_df['season'].astype(int)

train = model_df[model_df['season_int'] <= 2425]
test = model_df[model_df['season_int'] == 2526]

X_train, y_train, w_train = train[FEATURES], train['y'], train['recency_weight']
X_test, y_test = test[FEATURES], test['y']
print(f'Train: {len(train)} matches (2019-20 through 2024-25)')
print(f'Test : {len(test)} matches (2025-26)')
print()
print('Test season class balance:')
print(test['result'].value_counts().reindex(result_order))

Train: 839 matches (2019-20 through 2024-25)
Test : 140 matches (2025-26)

Test season class balance:
result
L    26
D    35
W    79
Name: count, dtype: int64


In [14]:
# walk-forward stability check: expanding window across season boundaries
walkforward_rows = []

for i in range(2, len(SEASON_ORDER)):
    tr_seasons = SEASON_ORDER[:i]
    te_season = SEASON_ORDER[i]
    tr = model_df[model_df['season_int'].isin(tr_seasons)]
    te = model_df[model_df['season_int'] == te_season]

    sc = StandardScaler()
    Xtr = sc.fit_transform(tr[FEATURES])
    Xte = sc.transform(te[FEATURES])

    m = LogisticRegression(max_iter = 2000)
    m.fit(Xtr, tr['y'], sample_weight = tr['recency_weight'])
    p = m.predict_proba(Xte)
    fold_logloss = log_loss(te['y'], p, labels=[0, 1, 2])

    train_priors = tr['y'].value_counts(normalize = True).reindex([0,1,2]).fillna(0).values
    fold_dummy_logloss = log_loss(te['y'], np.tile(train_priors, (len(te),1)), labels = [0,1,2])

    
    walkforward_rows.append({
        'test_season': te_season, 'n_train':len(tr), 'n_test':len(te),
        'logreg_logloss': fold_logloss, 'dummy_logloss': fold_dummy_logloss
    })

walkforward_df = pd.DataFrame(walkforward_rows)
walkforward_df['logreg_beats_dummy'] = walkforward_df['logreg_logloss'] < walkforward_df['dummy_logloss']
print(walkforward_df.round(4))

   test_season  n_train  n_test  logreg_logloss  dummy_logloss  \
0         2122      279     140          0.9040         0.9160   
1         2223      419     140          0.8681         0.9041   
2         2324      559     140          0.8409         0.9099   
3         2425      699     140          1.0081         1.0825   
4         2526      839     140          0.9615         0.9871   

   logreg_beats_dummy  
0                True  
1                True  
2                True  
3                True  
4                True  


### Reading Section F

---
## Section G - Baseline: Dummy Classifier

In [15]:
# DummyClassifier(strategy='prior'), log loss + accuracy floor
dummy = DummyClassifier(strategy = 'prior')
dummy.fit(X_train, y_train)
p_dummy = dummy.predict_proba(X_test)

dummy_logloss = log_loss(y_test, p_dummy, labels = [0,1,2])
dummy_acc = accuracy_score(y_test, dummy.predict(X_test))


print(f'Dummy baseline log loss: {dummy_logloss:.4f}')
print(f'Dummy baseline accuracy: {dummy_acc:.4f}')
print(f'(predicts P(L)={dummy.class_prior_[0]:.3f}, P(D)={dummy.class_prior_[1]:.3f}, P(W)={dummy.class_prior_[2]:.3f} for every single match)')

Dummy baseline log loss: 0.9871
Dummy baseline accuracy: 0.5643
(predicts P(L)=0.199, P(D)=0.209, P(W)=0.592 for every single match)


---
## Section H - Multinomial Logistic Regression

In [32]:
# StandardScaler, sklearn LogisticRegression, log loss / accuracy vs dummy
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=2000)
logreg.fit(X_train_s, y_train, sample_weight=w_train)
p_log = logreg.predict_proba(X_test_s)

log_logloss = log_loss(y_test, p_log, labels=[0, 1, 2])
log_acc = accuracy_score(y_test, logreg.predict(X_test_s))

print(f'Logistic regression log loss: {log_logloss:.4f}  (dummy: {dummy_logloss:.4f})')
print(f'Logistic regression accuracy: {log_acc:.4f}  (dummy: {dummy_acc:.4f})')

Logistic regression log loss: 0.9615  (dummy: 0.9871)
Logistic regression accuracy: 0.5643  (dummy: 0.5643)


In [34]:
# statsmodels MNLogit for coefficients + p-values, isolate stakes_intensity
Xc = sm.add_constant(pd.DataFrame(X_train_s, columns = FEATURES))
mnlogit = sm.MNLogit(y_train.values, Xc).fit(disp = 0)
print(mnlogit.summary())

                          MNLogit Regression Results                          
Dep. Variable:                      y   No. Observations:                  839
Model:                        MNLogit   Df Residuals:                      819
Method:                           MLE   Df Model:                           18
Date:                Thu, 17 Sep 2026   Pseudo R-squ.:                 0.07177
Time:                        16:52:39   Log-Likelihood:                -746.40
converged:                       True   LL-Null:                       -804.11
Covariance Type:            nonrobust   LLR p-value:                 3.041e-16
                y=1       coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.1646      0.120      1.372      0.170      -0.070       0.400
xG_roll5                0.0814      0.140      0.584      0.560      -0.192       0.355
xGA_roll5       

In [18]:
# confusion matrix


### Reading Section H

---
## Section I - Calibration

In [19]:
# calibration_curve for the Win class, reliability diagram


### Reading Section I

---
## Section J - Random Forest

In [20]:
# RandomForestClassifier, conservative depth given ~1000 rows
# log loss / accuracy vs logistic regression and dummy


---
## Section K - XGBoost

In [21]:
# XGBClassifier, conservative hyperparameters (shallow, regularized)


In [22]:
# summary table: dummy / logreg / rf / xgb -> log loss, accuracy, brier score


In [23]:
# walk-forward consistency check: does logistic regression beat the
# same-season dummy in every fold, not just the headline split


### Reading Section K

---
## Section L - SHAP Values

In [24]:
# TreeExplainer on the XGBoost model, mean |SHAP value| per feature
# for the Win class


In [25]:
# feature importance bar chart


### Reading Section L

---
## Key Findings Summary

---
## Bonus - Match-Level Walk-Forward Validation

Section F's evaluation used one model frozen before the 2025-26 season even started, trained once on everything through 2024-25 and never updated. A real deployed system would not work that way, it would retrain as results come in and use the freshest information available for the very next fixture. This section checks whether that actually matters.

Walking forward by team gameweek would be a mistake here, gameweek is computed per team and does not line up to the same calendar date across all 4 teams pooled in this dataset (a rescheduled fixture can push one team's "gameweek 5" onto a date another team's gameweek 8 has already passed). The correct unit to walk forward on is the actual match date, not gameweek number: for every distinct date a match was played in the 2025-26 season, train fresh on every row (any team, any season) dated strictly before it, then predict only what happened on that date.

In [26]:
# retrain before every distinct match date in 2025-26, predict only that date's matches
test_dates = sorted(model_df.loc[model_df['season'] == '2526', 'date'].unique())

wf_probs_list, wf_true_list = [], []
for d in test_dates:
    train_d = model_df[model_df['date'] < d]
    test_d = model_df[model_df['date'] == d]

    scaler_d = StandardScaler()
    Xtr = scaler_d.fit_transform(train_d[FEATURES])
    Xte = scaler_d.transform(test_d[FEATURES])

    model_d = LogisticRegression(max_iter=2000)
    model_d.fit(Xtr, train_d['y'], sample_weight=train_d['recency_weight'])

    wf_probs_list.append(model_d.predict_proba(Xte))
    wf_true_list.append(test_d['y'].values)

wf_probs = np.vstack(wf_probs_list)
wf_true = np.concatenate(wf_true_list)

wf_logloss = log_loss(wf_true, wf_probs, labels=[0, 1, 2])
wf_acc = accuracy_score(wf_true, wf_probs.argmax(axis=1))

print(f'Distinct match dates retrained on in 2025-26: {len(test_dates)}')
print(f'Walk-forward log loss: {wf_logloss:.4f}')
print(f'Walk-forward accuracy: {wf_acc:.4f}')

Distinct match dates retrained on in 2025-26: 120
Walk-forward log loss: 0.9641
Walk-forward accuracy: 0.5571


In [27]:
# same test matches, but scored by one model frozen before the season started
static_train = model_df[model_df['season'].astype(int) <= 2425]
static_test = model_df[model_df['season'] == '2526']

scaler_static = StandardScaler()
Xtr_static = scaler_static.fit_transform(static_train[FEATURES])
Xte_static = scaler_static.transform(static_test[FEATURES])

model_static = LogisticRegression(max_iter=2000)
model_static.fit(Xtr_static, static_train['y'], sample_weight=static_train['recency_weight'])
static_probs = model_static.predict_proba(Xte_static)

static_logloss = log_loss(static_test['y'], static_probs, labels=[0, 1, 2])
static_acc = accuracy_score(static_test['y'], static_probs.argmax(axis=1))

print(f'Static (frozen preseason) log loss: {static_logloss:.4f}')
print(f'Static (frozen preseason) accuracy: {static_acc:.4f}')
print()
print(f'Walk-forward log loss:              {wf_logloss:.4f}')
print(f'Walk-forward accuracy:              {wf_acc:.4f}')

Static (frozen preseason) log loss: 0.9615
Static (frozen preseason) accuracy: 0.5643

Walk-forward log loss:              0.9641
Walk-forward accuracy:              0.5571


### Reading the bonus section

120 distinct retrains across the 140 test matches, since `date` carries a full kickoff timestamp, not just a calendar day, this ended up close to genuine match-by-match walk-forward rather than the coarser weekend-batch version originally planned, and it stays leakage-safe either way since an earlier kickoff on the same weekend really did finish before a later one started.

The result is a mild surprise: walk-forward log loss (0.9641) and accuracy (0.5571) both come out slightly worse than the static frozen-preseason model (0.9615, 0.5643), not better. The gap is tiny, about one match's worth of accuracy across 140 test matches, so this isn't strong evidence that continuous retraining actively hurts. It is decent evidence that it does not help here. Each retrain only adds a handful of new rows to an already-large history (839+ matches), so the marginal information from "one more match happened" is small, while refitting the scaler and the logistic regression from scratch each time reintroduces a little extra variance for no real gain. For this feature set and this much history, one model fit once before the season does the same job as retraining 120 times through it, at a fraction of the cost.